In [1]:
import pandas as pd
import dotenv
from datetime import datetime, timedelta, timezone

dotenv.load_dotenv()

True

In [2]:
with open('data/mentalhealth_submissions.jsonl', 'r', encoding='utf-8') as f:
    for i in range(5):
        print(f.readline())

{"archived":true,"author":"robertduddeley","author_flair_background_color":null,"author_flair_css_class":null,"author_flair_richtext":[],"author_flair_text":null,"author_flair_text_color":null,"author_flair_type":"text","brand_safe":false,"can_gild":true,"contest_mode":false,"created_utc":1213296391,"distinguished":null,"domain":"addadhdblog.com","edited":false,"gilded":0,"hidden":false,"hide_score":false,"id":"6n4q9","is_crosspostable":true,"is_reddit_media_domain":false,"is_self":false,"is_video":false,"link_flair_css_class":null,"link_flair_richtext":[],"link_flair_text":null,"link_flair_text_color":"dark","link_flair_type":"text","locked":false,"media":null,"media_embed":{},"no_follow":true,"num_comments":0,"num_crossposts":0,"over_18":false,"parent_whitelist_status":null,"permalink":"\/r\/mentalhealth\/comments\/6n4q9\/women_with_add_are_more_impaired\/","retrieved_on":1522694976,"rte_mode":"markdown","score":1,"secure_media":null,"secure_media_embed":{},"selftext":"","send_replie

In [3]:
file_path = "data/mentalhealth_submissions.jsonl"
chunksize = 100_000  # tune for your RAM

# compute cutoff for "last year" from now
cutoff_dt = datetime.now(timezone.utc) - timedelta(days=365)
cutoff_ts = cutoff_dt.timestamp()

frames = []

for chunk in pd.read_json(file_path, lines=True, chunksize=chunksize):
    # keep only last 2 years
    chunk = chunk[chunk["created_utc"] >= cutoff_ts]

    # drop deleted/removed posts
    chunk = chunk[~chunk["selftext"].isin(["[deleted]", "[removed]"])]
    chunk = chunk[~chunk["author"].isin(["[deleted]", "[removed]"])]

    if len(chunk):
        frames.append(chunk)

time_filtered_df = pd.concat(frames, ignore_index=True)
clean_df = time_filtered_df[
    [
        "author",
        "selftext",
        "created_utc",
        "title",
        "permalink",
        "id",
        "subreddit",
        "subreddit_id",
        "link_flair_text",
        "retrieved_on",
    ]
].dropna()

# add t3_ prefix to id and rename to post_id
# chromadb uses "id" to identify documents
clean_df["post_id"] = "t3_" + clean_df["id"].astype(str)
clean_df = clean_df.drop(columns=["id"])

print(len(clean_df), "rows in last 1 years after filtering")

98960 rows in last 1 years after filtering


## Inspecting Data

Huh. Turns out there's a lot of entries with no text. Let's get rid of them. Only keep entries with 100+ words.

In [4]:
selftext_lengths = clean_df["selftext"].fillna("").apply(lambda x: len(str(x).split()))
title_lengths = clean_df["title"].fillna("").apply(lambda x: len(str(x).split()))

zero_len_df = clean_df[selftext_lengths == 0]

print(f"Number of posts with 0-word selftext: {len(zero_len_df)}")
print(zero_len_df[["author", "title", "selftext", "permalink"]].head(5).to_string(index=False))

Number of posts with 0-word selftext: 909
              author                                                                        title selftext                                                                           permalink
  Infinite_Word_8211                                                     reconnecting with nature                                   /r/mentalhealth/comments/1j9kcr3/reconnecting_with_nature/
      universityofga Vacations are good for employee well-being, and the effects are long lasting          /r/mentalhealth/comments/1j9ped1/vacations_are_good_for_employee_wellbeing_and_the/
 whotfareyoutalkinto                                     random positivity found on youtube today                   /r/mentalhealth/comments/1ja2ttm/random_positivity_found_on_youtube_today/
  plsnomoresuffering                                          Following the heart purges the mind                        /r/mentalhealth/comments/1jal1gj/following_the_heart_purges_the_mind/
Inf

In [5]:
clean_df = clean_df[selftext_lengths > 100]
print(f"{len(clean_df)} rows remain after dropping posts with 100-word selftext")

68175 rows remain after dropping posts with 100-word selftext


 turns chromadb limits document sizes to 16kb and we have text documents in here that's larger than 16kb.

we wil find out whats the max length of a title (since our document combines title with body text), and then set a cap to how long the body can be, then truncate

In [6]:
max_title_len = clean_df["title"].fillna("").apply(lambda x: len(str(x))).max()
print(f"Maximum title length in characters: {max_title_len}")

Maximum title length in characters: 300


In [7]:
# ChromaDB limits documents to 16KB (~16k chars). Truncate selftext so title + " | " + selftext <= 16000
MAX_DOC_CHARS = 16000
SEPARATOR = " | "

before_len = clean_df["title"].fillna("").str.len() + len(SEPARATOR) + clean_df["selftext"].fillna("").str.len()
n_truncated = (before_len > MAX_DOC_CHARS).sum()

def truncate_selftext(row):
    title = str(row["title"]) if pd.notna(row["title"]) else ""
    selftext = str(row["selftext"]) if pd.notna(row["selftext"]) else ""
    max_selftext_len = MAX_DOC_CHARS - len(title) - len(SEPARATOR)
    if len(selftext) > max_selftext_len:
        return selftext[:max_selftext_len]
    return selftext

clean_df["selftext"] = clean_df.apply(truncate_selftext, axis=1)
print(f"Documents truncated to {MAX_DOC_CHARS} chars. Rows affected: {n_truncated}")

# Show an example of a truncated document
if n_truncated > 0:
    example_idx = before_len[before_len > MAX_DOC_CHARS].index[0]
    row = clean_df.loc[example_idx]
    doc = row["title"] + SEPARATOR + row["selftext"]
    print(f"\nExample truncated doc (post_id={row['post_id']}, orig length={int(before_len.loc[example_idx])} chars):")
    print(f"  Total length now: {len(doc)} chars")
    print(f"  Title: {row['title'][:80]}...")
    print(f"  Selftext (last 400 chars, where cut): ...{row['selftext'][-400:]}")

Documents truncated to 16000 chars. Rows affected: 0


## Collection Size modification

In [8]:
subsample_df = clean_df.sample(n=70000, random_state=67) if len(clean_df) > 70000 else clean_df

len(subsample_df)

68175

## Cost and Size Checks

In [21]:
import tiktoken

encoding = tiktoken.encoding_for_model("text-embedding-3-small")

def count_selftext_tokens(df, encoding):
    total_tokens = 0
    titles = df["title"].fillna("")
    selftexts = df["selftext"].fillna("")

    for i, (title, selftext) in enumerate(zip(titles, selftexts), 1):
        text = f"{title} | {selftext}"
        num_tokens = len(encoding.encode(str(text)))
        total_tokens += num_tokens
        if i % 20000 == 0:
            print(f"Token sum after {i} rows: {total_tokens:,}")

    print(f"Total tokens in df title|selftext: {total_tokens:,}")
    # openai text-embedding-3-small cost is $0.02 per million tokens
    cost = total_tokens / 1000000 * 0.02
    print(f"Cost: ${cost:.2f}")
    return

count_selftext_tokens(subsample_df, encoding)


Token sum after 20000 rows: 6,691,584
Token sum after 40000 rows: 13,456,166
Token sum after 60000 rows: 20,160,041
Total tokens in df title|selftext: 22,938,583
Cost: $0.46


In [10]:
BYTES_PER_VALUE = 4   # float32


def estimate_vector_store_size(num_vectors, dim, bytes_per_value=BYTES_PER_VALUE):
    total_bytes = num_vectors * dim * bytes_per_value
    total_mb = total_bytes / (1024 ** 2)
    total_gb = total_bytes / (1024 ** 3)
    print(f"Estimated raw embedding storage for {num_vectors:,} vectors:")
    print(f"~{total_mb:,.2f} MB (~{total_gb:,.2f} GB) of float32 embeddings")
    return


num_vectors = len(subsample_df)
estimate_vector_store_size(num_vectors, 1536)
estimate_vector_store_size(num_vectors, 512)
estimate_vector_store_size(num_vectors, 256)

Estimated raw embedding storage for 68,175 vectors:
~399.46 MB (~0.39 GB) of float32 embeddings
Estimated raw embedding storage for 68,175 vectors:
~133.15 MB (~0.13 GB) of float32 embeddings
Estimated raw embedding storage for 68,175 vectors:
~66.58 MB (~0.07 GB) of float32 embeddings


## Supabase setup

In [2]:
import vecs
import os
import dotenv
dotenv.load_dotenv()
DB_CONNECTION = os.environ.get("DB_CONNECTION")

vx = vecs.create_client(DB_CONNECTION)

In [3]:
docs = vx.get_or_create_collection(name="docs", dimension=1536)

In [4]:
from openai import OpenAI

dotenv.load_dotenv()

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),
)


def create_embeddings(input):
    response = client.embeddings.create(
    model="text-embedding-3-small",
    input=input,
    encoding_format="float"
    )
    return response.data[0].embedding

## Upserting test

In [25]:
test_subsample_df = subsample_df.head(3)



In [26]:
def create_embeddings_batch(texts):
    if not texts:
        return []
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts,
        encoding_format="float",
    )
    return [d.embedding for d in response.data]


def sanitize_metadata(row):
    """Vecs metadata: JSON-serializable primitives only (str, int, float, bool)."""
    return {
        "author": str(row["author"]) if pd.notna(row["author"]) else "",
        "created_utc": int(row["created_utc"]) if pd.notna(row["created_utc"]) else 0,
        "permalink": str(row["permalink"]) if pd.notna(row["permalink"]) else "",
        "subreddit": str(row["subreddit"]) if pd.notna(row["subreddit"]) else "",
        "subreddit_id": str(row["subreddit_id"]) if pd.notna(row["subreddit_id"]) else "",
        "link_flair_text": str(row["link_flair_text"]) if pd.notna(row["link_flair_text"]) else "",
        "retrieved_on": int(row["retrieved_on"]) if pd.notna(row["retrieved_on"]) else 0,
        "post_id": str(row["post_id"]),
        "title": str(row["title"]),
        "selftext": str(row["selftext"])
    }


BATCH_SIZE = 100
df_to_upsert = test_subsample_df  # or clean_df

batch_df = df_to_upsert.head(BATCH_SIZE)
documents = (batch_df["title"] + " | " + batch_df["selftext"]).tolist()
embeddings = create_embeddings_batch(documents)

records = []
for i, (_, row) in enumerate(batch_df.iterrows()):
    post_id = str(row["post_id"])
    vec = embeddings[i]
    meta = sanitize_metadata(row)
    records.append((post_id, vec, meta))

docs.upsert(records=records)
print(f"Upserted {len(records)} records (ids = post_id).")

Upserted 3 records (ids = post_id).


In [27]:
docs.create_index(
    method=vecs.IndexMethod.hnsw,
    measure=vecs.IndexMeasure.cosine_distance
)

## Upsert to vecs with custom embeddings

Use `post_id` as the vector id. We create embeddings ourselves with OpenAI, then `docs.upsert(records=[(id, vector, metadata), ...])`.

In [30]:
# Full dataset: batch upsert with progress (resumable)
VECS_BATCH_SIZE = 100
vecs_progress_file = "data/vecs_upload_progress.txt"
df_to_upsert = clean_df

def load_vecs_progress(default_start: int = 0) -> int:
    if not os.path.exists(vecs_progress_file):
        return default_start
    with open(vecs_progress_file, "r") as f:
        return int(f.read().strip())


def save_vecs_progress(next_index: int) -> None:
    with open(vecs_progress_file, "w") as f:
        f.write(str(next_index))


start_idx = load_vecs_progress()
n_total = len(df_to_upsert)
print(f"Starting vecs upsert from index {start_idx} of {n_total}, batch size={VECS_BATCH_SIZE}.")

for batch_start in range(start_idx, n_total, VECS_BATCH_SIZE):
    batch_end = min(batch_start + VECS_BATCH_SIZE, n_total)
    batch_df = df_to_upsert.iloc[batch_start:batch_end]
    documents = (batch_df["title"] + " | " + batch_df["selftext"]).tolist()
    embeddings = create_embeddings_batch(documents)
    records = []
    for i, (_, row) in enumerate(batch_df.iterrows()):
        records.append((str(row["post_id"]), embeddings[i], sanitize_metadata(row)))
    try:
        docs.upsert(records=records)
        save_vecs_progress(batch_end)
        print(f"Upserted batch {batch_start}:{batch_end}.")
    except Exception as e:
        print(f"Error at batch {batch_start}:{batch_end}: {e}")
        break

Starting vecs upsert from index 3 of 68175, batch size=100.
Upserted batch 3:103.
Upserted batch 103:203.
Upserted batch 203:303.
Upserted batch 303:403.
Upserted batch 403:503.
Upserted batch 503:603.
Upserted batch 603:703.
Upserted batch 703:803.
Upserted batch 803:903.
Upserted batch 903:1003.
Upserted batch 1003:1103.
Upserted batch 1103:1203.
Upserted batch 1203:1303.
Upserted batch 1303:1403.
Upserted batch 1403:1503.
Upserted batch 1503:1603.
Upserted batch 1603:1703.
Upserted batch 1703:1803.
Upserted batch 1803:1903.
Upserted batch 1903:2003.
Upserted batch 2003:2103.
Upserted batch 2103:2203.
Upserted batch 2203:2303.
Upserted batch 2303:2403.
Upserted batch 2403:2503.
Upserted batch 2503:2603.
Upserted batch 2603:2703.
Upserted batch 2703:2803.
Upserted batch 2803:2903.
Upserted batch 2903:3003.
Upserted batch 3003:3103.
Upserted batch 3103:3203.
Upserted batch 3203:3303.
Upserted batch 3303:3403.
Upserted batch 3403:3503.
Upserted batch 3503:3603.
Upserted batch 3603:3703.

In [7]:
post_query = """
This will be a long post so bear with me. I’m writing this all down for therapeutic purposes. But I would like some input on what was happening here and what to do at this point.

When I was in high school, I had a science teacher for two years (sophomore and junior year) and ended up becoming very close to him. Even though I wouldn’t admit it to anyone, not even myself, I was infatuated with him and experiencing limerence. He was very charming, and with me not having my father in the picture (my dad was an absent alcoholic) I latched on to him as a father figure. He was a good married man with three children. Sometimes he would make comments on how “petite” I was (I am a man) and touch my hair and shoulders. I deemed it as just paternal affection. I so desperately craved it, as I was going through a lot at home.

By senior year, him and I had a pretty profound rapport, and he even paid for me to go a school field trip in San Francisco with him and other students. When I graduated I asked for his number and we stayed in touch from there.

I went to college in NYC and there was a moment where he felt very involved and responsive even though I was away at school. At this point, I was very much in love with him (I could not admit it to myself yet). I would come back to Long Island regularly to visit him (I remember he affectionately touched my thigh while we got lunch together)

Anyway it all came to a head when he came out to the city to see me. He showed up at my dorm room and we eventually got some drinks together at a hookah bar. I remember he took his shoes off and put his feet right next to my lap. That’s about it. But from that point on he gradually withdrew. He became less responsive. While this was going on I started to drink heavily and eventually became an alcoholic.

When I was 20 (2012) I confessed to him that I was in love with him, and that I couldn’t talk to him for a while. His response was that I was a beautiful person and it was ok. A few months later we started talking again but by then my drinking was so bad that I threatened to kill myself over all of this and cursed him out. He promptly cut me out of his life. For years and years, any message I left from time to time was unanswered. Part of me understands why but another part of me felt it was so brutal and I was once again abandoned by someone I really loved.

Fast forward to 2021 - This same teacher is escorted out of school for disturbing allegations about his relationships with current/former female students. He was fired from the school.

This is now a long time ago, but to this day, I wonder where I fit into all of that. I am in a lovely relationship with a woman now, have a decent career, am nine years sober. I’ve moved on. But yet all of this still haunts me. I lost years of my life due to my attachment to him. I began to do sex work after he cut me out of his life and ended up getting sexually assaulted a few times.

After he was fired I reached out to him and he made an amends for cutting me off like that but that’s about it. I was selfishly trying to have him as an acquaintance again for “closure” but decided to cut him out of my life because the last thing I need is to regress emotionally. And what he did to those girls is reprehensible.

Was there abuse on his end? Or was I just crazy? Any feedback on this would be appreciated. Thank you.
"""

In [8]:
query_embeddings = create_embeddings(post_query)
documents = docs.query(
    data=query_embeddings,          
    limit=10,
    include_metadata=True,
)
for doc, metadata in documents:
    print(metadata["permalink"])

/r/mentalhealth/comments/1lh9fjf/how_to_mentally_deal_with_this/
/r/mentalhealth/comments/1lifsgs/i_was_saed_by_a_man/
/r/mentalhealth/comments/1mmpzm1/my_ex_wants_to_talk/
/r/mentalhealth/comments/1k99k68/please_help_me_untangle_this_emotional_mess/
/r/mentalhealth/comments/1ll79r1/still_struggling_with_guilt_and_trauma_after/
/r/mentalhealth/comments/1k906pk/my_ex_emotionally_destroyed_me/
/r/mentalhealth/comments/1kkiryp/i_need_some_advise/
/r/mentalhealth/comments/1mas2lx/not_sure_if_im_being_coercedabused_or_not_and_its/
/r/mentalhealth/comments/1jlrs1b/i_believe_i_was_groomed_for_4_years_by_an_older/
/r/mentalhealth/comments/1mop3e9/realizing_that_i_was_potentially_emotionally/


# Create ChromaDB

In [98]:
import chromadb
import os

client = chromadb.CloudClient(
  api_key=os.environ.get("CHROMA_API_KEY"),
)

In [ ]:
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

COLLECTION_NAME = "reddit-bot-vdb-2"

collection = client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=OpenAIEmbeddingFunction(
        model_name="text-embedding-3-small",
        api_key=os.environ.get("OPENAI_API_KEY"),
    ),
)

collection = client.get_collection(COLLECTION_NAME)

## Small batch test upload to chroma

In [70]:
chroma_test_df = clean_df.head(3)

In [102]:

documents = (
    chroma_test_df["title"]
    + " | "
    + chroma_test_df["selftext"]
).tolist()

ids = chroma_test_df["post_id"].astype(str).tolist()

metadatas = chroma_test_df[
    [
        "author",
        "created_utc",
        "permalink",
        "subreddit",
        "subreddit_id",
        "link_flair_text",
        "retrieved_on",
        "post_id",   
    ]
].to_dict(orient="records")

print(documents[:1])
print(ids[:3])
print(metadatas[:2])

["The Junkie, his ADHD, and his Moment of Clarity | I took to self-medicating at 14, long before my ADHD diagnosis at 23. By age 19, I had already been a chronic teenage alcoholic and graduated as a seasoned meth user by age 20. After years of inpatients, outpatients, slips, and relapses, I couldn't figure out what the hell was wrong with me. I tried the 12 steps, S.M.A.R.T. Recovery, health realization, etc... over and over, to no avail. I just couldn't stay sober and tried everything before I surrendered to the idea that complete abstinence just wasn't my thing. Keep in mind that I was still undiagnosed at this point.  \nHarm reduction is controversial. Some see it as swapping one bad habit out for another, while others advocate for it. I had no clue where to turn before my diagnosis, and I knew abstinence wasn't going to work. After some research, I finally found an alternative rehab that focused on co-occurring disorders as opposed to mere drug abuse. This place saved my fuckin' li

In [103]:
collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
)

C:\Users\moust\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:08<00:00, 9.91MiB/s]


## Batch upload to safeguard against network errors/api timeout

In [96]:
clean_df.to_csv("data/clean_df.csv", index=False)

In [104]:
BATCH_SIZE = 300
progress_file = "data/chroma_upload_progress.txt"


def load_progress(default_start: int = 0) -> int:
    """Return the next row index to start from.

    If the progress file does not exist, start at `default_start`.
    If it exists, it should contain a single integer: the next index.
    """
    if not os.path.exists(progress_file):
        return default_start

    with open(progress_file, "r") as f:
        value = f.read().strip()
        return int(value)



def save_progress(next_index: int) -> None:
    with open(progress_file, "w") as f:
        f.write(str(next_index))


In [105]:
start = load_progress()
n = len(clean_df)

print(f"Starting upload from index {start} out of {n} rows, batch size={BATCH_SIZE}.")

for batch_start in range(start, n, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, n)
    print(f"Preparing batch {batch_start}:{batch_end} (size={batch_end - batch_start})")

    batch_df = clean_df.iloc[batch_start:batch_end]

    ids = batch_df["post_id"].astype(str).tolist()
    documents = (batch_df["title"] + " | " + batch_df["selftext"]).tolist()
    metadatas = batch_df[
        [
            "author",
            "created_utc",
            "permalink",
            "subreddit",
            "subreddit_id",
            "link_flair_text",
            "retrieved_on",
            "post_id",
        ]
    ].to_dict(orient="records")

    try:
        print(f"Uploading batch starting at index {batch_start} with {len(ids)} documents...")
        collection.add(ids=ids, documents=documents, metadatas=metadatas)
        print(f"Successfully uploaded batch {batch_start}:{batch_end}.")
    except Exception as e:
        print(f"Error uploading batch {batch_start}:{batch_end}: {e}")
        break

    save_progress(batch_end)
    print(f"Progress saved. Next start index will be {batch_end}.")


Starting upload from index 0 out of 143680 rows, batch size=300.
Preparing batch 0:300 (size=300)
Uploading batch starting at index 0 with 300 documents...
Successfully uploaded batch 0:300.
Progress saved. Next start index will be 300.
Preparing batch 300:600 (size=300)
Uploading batch starting at index 300 with 300 documents...
Successfully uploaded batch 300:600.
Progress saved. Next start index will be 600.
Preparing batch 600:900 (size=300)
Uploading batch starting at index 600 with 300 documents...
Successfully uploaded batch 600:900.
Progress saved. Next start index will be 900.
Preparing batch 900:1200 (size=300)
Uploading batch starting at index 900 with 300 documents...
Successfully uploaded batch 900:1200.
Progress saved. Next start index will be 1200.
Preparing batch 1200:1500 (size=300)
Uploading batch starting at index 1200 with 300 documents...
Successfully uploaded batch 1200:1500.
Progress saved. Next start index will be 1500.
Preparing batch 1500:1800 (size=300)
Uploa